In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

In [ ]:
df = pd.read_csv('data/SRP100712.tsv', sep='\t')
df.head()


In [ ]:
print(df.shape)

32309 Genes with 594 samples

In [ ]:
meta = pd.read_csv('data/metadata_SRP100712.tsv', sep='\t')
meta = meta.set_index("refinebio_accession_code")
meta = meta.loc[df.drop(columns=["Gene"]).columns]  # align to expression columns

meta["condition"] = np.where(meta["refinebio_subject"].str.contains("spaceflight"), "flight", "ground")
meta["age"] = meta["refinebio_age"].astype(int).astype(str) + "d"

print("Unique sample titles:", meta["refinebio_title"].nunique())

In [ ]:
drop_gene = df.drop(columns=["Gene"])
drop_gene.index = df["Gene"]  # keep gene IDs as the index instead of losing them

drop_gene = drop_gene.T.groupby(meta["refinebio_title"]).sum().T
print(drop_gene.shape)
# should now print (32309, 32)

log_expr = np.log2(drop_gene + 1)
gene_medians = log_expr.median(axis=1)
gene_std = log_expr.std(axis=1)

print(gene_medians.describe())
print("Average per-gene std:", gene_std.mean())

plt.figure(figsize=(7, 4))
sns.kdeplot(gene_medians, fill=True)
plt.xlabel("Per-gene median log2(expression + 1)")
plt.ylabel("Density")
plt.title("Distribution of per-gene median expression")
plt.savefig("results/median_density.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from umap import UMAP


log_expression = log_expr.T  # transpose to have samples as rows and genes as columns

pca = PCA(n_components=2)
tsne = TSNE(n_components=2, perplexity=8,random_state=42)
umap = UMAP(n_components=2, n_neighbors=10, min_dist=0.1, random_state=42)

In [ ]:
pca_result = pca.fit_transform(log_expression)
tsne_result = tsne.fit_transform(log_expression)
umap_result = umap.fit_transform(log_expression)

In [ ]:
print("pca_result:", len(pca_result))
print("tsne_result:", len(tsne_result))
print("umap_result:", len(umap_result))

In [ ]:
with open("data/aggregated_metadata.json", "r") as f:
    metadata = json.load(f)

samples = metadata["samples"]
conditions = {}

for sample_id, info in samples.items():
    title = info["refinebio_title"]
    if title.startswith("Spaceflight"):
        conditions[sample_id] = "Spaceflight"
    elif title.startswith("Ground Control"):
        conditions[sample_id] = "Ground Control"

condition_series = meta.groupby(meta["refinebio_title"])["condition"].first()
condition_series = condition_series.loc[log_expr.columns]  # match the order used for PCA/tSNE/UMAP
print(len(condition_series))  # should now print 32

In [ ]:
pca_df = pd.DataFrame({
    "PC1": pca_result[:, 0],
    "PC2": pca_result[:, 1],
    "Condition": condition_series.values
})

tsne_df = pd.DataFrame({
    "tSNE1": tsne_result[:, 0],
    "tSNE2": tsne_result[:, 1],
    "Condition": condition_series.values
})

umap_df = pd.DataFrame({
    "UMAP1": umap_result[:, 0],
    "UMAP2": umap_result[:, 1],
    "Condition": condition_series.values
})

In [ ]:
plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="Condition",
    s=80
)

plt.xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% variance)"
)
plt.ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% variance)"
)

In [ ]:
plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=tsne_df,
    x="tSNE1",
    y="tSNE2",
    hue="Condition",
    s=80
)

plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("t-SNE of RNA-seq Samples")

plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=umap_df,
    x="UMAP1",
    y="UMAP2",
    hue="Condition",
    s=80
)

plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.title("UMAP of RNA-seq Samples")

plt.show() 

In [ ]:
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

In [ ]:
sample_meta = meta.groupby("refinebio_title")[["condition", "age"]].first()

sample_meta = sample_meta.loc[log_expr.columns]

sample_meta["condition"].value_counts()

In [ ]:
flight = sample_meta.index[sample_meta["condition"] == "flight"]
ground = sample_meta.index[sample_meta["condition"] == "ground"]

In [ ]:
flight_mean = log_expr[flight].mean(axis=1)
ground_mean = log_expr[ground].mean(axis=1)

effect = flight_mean - ground_mean

In [ ]:
p_values = []

for gene in log_expr.index:
    test = ttest_ind(
        log_expr.loc[gene, flight],
        log_expr.loc[gene, ground],
        equal_var=False
    )

    p_values.append(test.pvalue)

In [ ]:
results = pd.DataFrame({
    "Gene": log_expr.index,
    "Flight_Mean": flight_mean.values,
    "Ground_Mean": ground_mean.values,
    "Effect": effect.values,
    "P_Value": p_values
})

In [ ]:
results["P_Value"] = results["P_Value"].fillna(1.0)

results["Adjusted_P_Value"] = multipletests(
    results["P_Value"],
    method="fdr_bh"
)[1]

In [ ]:
results = results.sort_values("Adjusted_P_Value")
results.head()

In [ ]:
results["Significant"] = results["Adjusted_P_Value"] < 0.05

results["Significant"].value_counts()

In [ ]:
results["-log10_p"] = -np.log10(results["Adjusted_P_Value"].clip(lower=1e-300))

In [ ]:
plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=results,
    x="Effect",
    y="-log10_p",
    hue="Significant"
)

plt.xlabel("Expression Difference (Flight - Ground)")
plt.ylabel("-log10 Adjusted P-Value")
plt.title("Volcano Plot: Flight vs Ground")
plt.xlim(-10, 10)
plt.legend()

plt.show()

In [ ]:
top50 = results.head(50)

top50

In [ ]:
results.to_csv(
    "results/differential_expression_all.csv",
    index=False
)

top50.to_csv(
    "results/differential_expression_top50.csv",
    index=False
)

In [ ]:
sig_genes = results[
    results["Adjusted_P_Value"] < 0.05
]["Gene"]

In [ ]:
heatmap_data = log_expr.loc[sig_genes]

In [ ]:
group_colors = sample_meta["condition"].map({
    "flight": "red",
    "ground": "blue"
})

In [ ]:
sns.clustermap(
    heatmap_data,
    col_colors=group_colors,
    cmap="vlag",
    xticklabels=False,
    yticklabels=False
)

plt.show()

In [ ]:
import mygene

mg = mygene.MyGeneInfo()

In [ ]:
genes = results["Gene"].tolist()

annotations = mg.querymany(
    genes,
    scopes="symbol",
    fields="go.BP",
    species=3702
)

In [ ]:
go_rows = []

for item in annotations:
    gene = item["query"]

    if "go" in item and "BP" in item["go"]:
        terms = item["go"]["BP"]

        if isinstance(terms, dict):
            terms = [terms]

        for term in terms:
            go_rows.append([
                gene,
                term["id"],
                term["term"]
            ])

In [ ]:
go = pd.DataFrame(
    go_rows,
    columns=["Gene", "GO_ID", "GO_term"]
)

go = go.drop_duplicates()

go = go[
    go["GO_ID"] != "GO:0008150"
]

go.head()

In [ ]:
go.to_csv(
    "results/arabidopsis_go_bp_annotations.csv",
    index=False
)

In [ ]:
go_data = go.merge(
    results[["Gene", "Effect"]],
    on="Gene"
)

In [ ]:
from scipy.stats import mannwhitneyu

enrichment_results = []

for go_id, group in go_data.groupby("GO_ID"):
    inside = group["Effect"]

    outside = results[
        ~results["Gene"].isin(group["Gene"])
    ]["Effect"]

    if len(inside) >= 5:
        p = mannwhitneyu(
            inside,
            outside
        ).pvalue

        enrichment_results.append([
            go_id,
            group["GO_term"].iloc[0],
            len(inside),
            p
        ])

In [ ]:
enrichment = pd.DataFrame(
    enrichment_results,
    columns=[
        "GO_ID",
        "GO_term",
        "Gene_Count",
        "P_Value"
    ]
)

In [ ]:
enrichment["Adjusted_P_Value"] = multipletests(
    enrichment["P_Value"],
    method="fdr_bh"
)[1]

enrichment = enrichment.sort_values(
    "Adjusted_P_Value"
)

enrichment.head(20)

In [ ]:
enrichment.to_csv(
    "results/wilcoxon_GO_BP_enrichment.csv",
    index=False
)

In [ ]:
print("Genes tested:", results["Gene"].nunique())
print("Genes with GO annotations:", go["Gene"].nunique())